In [7]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import os

input_csv = "../data/competitions.csv"
output_csv = "../data/competition_logos.csv"

# ✅ Store already scraped values by competition_id
existing_data = {}

if os.path.exists(output_csv):
    with open(output_csv, newline='', encoding='utf-8') as existing_file:
        reader = csv.DictReader(existing_file)
        for row in reader:
            existing_data[row['competition_id']] = {
                'cup_image_url': row.get('cup_image_url', '').strip(),
                'competition_logo_url': row.get('competition_logo_url', '').strip()
            }

# Open input and output files
with open(input_csv, newline='', encoding='utf-8') as infile, \
     open(output_csv, mode='a', newline='', encoding='utf-8') as outfile:

    reader = csv.DictReader(infile)
    fieldnames = ['competition_id', 'competition_name', 'cup_image_url', 'competition_logo_url']
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    if os.stat(output_csv).st_size == 0:
        writer.writeheader()

    for row in reader:
        comp_id = row['competition_id'].strip()
        comp_name = row['name'].strip()
        url = row['url'].strip()

        existing = existing_data.get(comp_id, {})
        cup_exists = bool(existing.get('cup_image_url'))
        logo_exists = bool(existing.get('competition_logo_url'))

        if cup_exists and logo_exists:
            print(f"⏭️ Already has both images: {comp_name} ({comp_id})")
            continue

        print(f"🔍 Scraping {comp_name} ({comp_id})...")
        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(url, headers=headers, timeout=30)

            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')

                def full_url(src):
                    if src.startswith("//"):
                        return "https:" + src
                    elif src.startswith("/"):
                        return "https://www.transfermarkt.com" + src
                    return src

                cup_img_tag = soup.find('img', src=lambda src: src and "/images/erfolge/fix/" in src)
                logo_img_tag = soup.find('img', src=lambda src and "/images/logo/header/" in src)

                cup_url = existing.get('cup_image_url') or (full_url(cup_img_tag['src']) if cup_img_tag else '')
                logo_url = existing.get('competition_logo_url') or (full_url(logo_img_tag['src']) if logo_img_tag else '')

                writer.writerow({
                    'competition_id': comp_id,
                    'competition_name': comp_name,
                    'cup_image_url': cup_url,
                    'competition_logo_url': logo_url
                })

                print(f"✅ Cup: {cup_url if cup_url else '❌ Not found'}, Logo: {logo_url if logo_url else '❌ Not found'}")

            else:
                print(f"❌ Failed to fetch {url} (status {response.status_code})")

        except Exception as e:
            print(f"❌ Error scraping {comp_name}: {e}")

        time.sleep(1)  # Be polite


SyntaxError: invalid syntax (712054445.py, line 62)